# LLM Exception Handling Evaluation Analysis
This notebook analyzes the performance of various LLMs on Exception Handling code generation tasks (Tasks 1-4). It processes `metrics.csv` for aggregate performance, `task4_quality_details.csv` for fine-grained snippet analysis, and explores token usage efficiency.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)

# Paths
RESULTS_DIR = "/home/tales/Mestrado/exception-miner-multi/llm/results"
METRICS_FILE = "/home/tales/Mestrado/exception-miner-multi/llm/output/metrics.csv"
DETAILS_FILE = "/home/tales/Mestrado/exception-miner-multi/llm/output/task4_quality_details.csv"

def add_bar_labels(ax, fmt='%.2f', padding=3):
    """Helper function to add labels to bar charts"""
    for container in ax.containers:
        ax.bar_label(container, fmt=fmt, label_type='edge', padding=padding)


## 1. Load and Inspect Aggregate Metrics

In [ ]:
metrics_df = pd.read_csv(METRICS_FILE)
display(metrics_df.head())

# Pivot the table for easier comparison across models and styles
pivot_metrics = metrics_df.pivot_table(
    index=["task", "model", "style-prompt"],
    columns="metric",
    values="value"
).reset_index()
display(pivot_metrics.head())


### Compare CrystalBLEU Scores (Task 4)

In [ ]:
t4_metrics = metrics_df[(metrics_df["task"] == "task4") & (metrics_df["metric"] == "CrystalBLEU")]

plt.figure(figsize=(14, 7))
ax = sns.barplot(data=t4_metrics, x="model", y="value", hue="style-prompt")
plt.title("Task 4: CrystalBLEU Score by Model and Prompt Style")
plt.ylabel("CrystalBLEU Score")
plt.xlabel("Model")
plt.xticks(rotation=45)
add_bar_labels(ax, fmt='%.3f')
plt.tight_layout()
plt.show()


## 2. Evaluate Task 3: Exception Hierarchy
Comparing Exact Accuracy vs Hierarchy Accuracy to see how often models get the category right even if they miss the exact exception.
Also specifically looking at `gpt-4o-mini` with `few-shot` style as requested.

In [ ]:
t3_metrics = metrics_df[(metrics_df["task"] == "task3") & (metrics_df["metric"].isin(["Exact Accuracy", "Hierarchy Accuracy"]))]

plt.figure(figsize=(14, 7))
ax = sns.barplot(data=t3_metrics, x="model", y="value", hue="metric", errorbar=None)
plt.title("Task 3: Exact vs. Hierarchy Accuracy (All Models)")
plt.ylabel("Accuracy (0.0 to 1.0)")
plt.xticks(rotation=45)
plt.legend(title="Metric")
add_bar_labels(ax, fmt='%.2f')
plt.tight_layout()
plt.show()

# Specific analysis for gpt-4o-mini few-shot
print("Specific Comparison for GPT-4o-Mini (Few-Shot):")
gpt4o_fs = t3_metrics[(t3_metrics["model"] == "gpt-4o-mini") & (t3_metrics["style-prompt"] == "style-few-shot")]
display(gpt4o_fs)


## 3. Analyze Task 4 Static Analysis (Pylint Violations)
Comparing the average violations generated by the LLM versus the original ground truth (human) code.

In [ ]:
violation_metrics = metrics_df[(metrics_df["task"] == "task4") & (metrics_df["metric"].isin(["Avg Violations", "GT Avg Violations"]))]

plt.figure(figsize=(14, 7))
ax = sns.barplot(data=violation_metrics, x="model", y="value", hue="metric", errorbar=None)
plt.title("Task 4: Average Pylint Violations (LLM vs Ground Truth)")
plt.ylabel("Avg Violations per Snippet")
plt.xticks(rotation=45)
add_bar_labels(ax, fmt='%.2f')
plt.tight_layout()
plt.show()


## 4. Deep Dive: Task 4 Quality Details
Let's look at the specific anti-patterns the models are generating, showing absolute numbers and percentages.

In [ ]:
details_df = pd.read_csv(DETAILS_FILE)

# Calculate the percentage of perfectly matching snippets
exact_match_pct = details_df.groupby(["model", "style"])["exact_match"].mean() * 100
print("Exact Match Percentage (%) by Model and Style:")
display(exact_match_pct.unstack().round(2))

# Find the most common violations generated by LLMs
all_llm_violations = details_df["pred_violation_codes"].dropna().str.split(",").explode()
all_llm_violations = all_llm_violations[all_llm_violations != ""] # Remove empty strings
violation_counts = all_llm_violations.value_counts()
violation_pct = (violation_counts / len(details_df) * 100).round(2)

plt.figure(figsize=(10, 6))
ax = violation_counts.head(10).plot(kind="bar", color="salmon")
plt.title("Most Common Pylint Violations by LLMs (Total Count)")
plt.ylabel("Count of Violations")
plt.xlabel("Pylint Rule Code")
add_bar_labels(ax, fmt='%d')
plt.tight_layout()
plt.show()

print("Violation Frequencies (Count and Percentage of total snippets):")
violation_summary = pd.DataFrame({"Count": violation_counts, "Percentage (%)": violation_pct})
display(violation_summary.head(10))


## 5. Token Information vs. Performance (Cost/Efficiency Analysis)
We extract the token usage from the raw `_results_all.csv` files. This allows us to see how many tokens each task and prompt style consumes, helping evaluate if expensive prompt styles (like CoT) are worth it.

In [ ]:
# Load raw results to get token usage
all_result_files = glob.glob(os.path.join(RESULTS_DIR, "*_results_all.csv"))

token_data = []
for file in all_result_files:
    try:
        df = pd.read_csv(file)
        # Extract model from filename (e.g., combined_gpt-4o-mini_results_all.csv)
        model_name = os.path.basename(file).split("_")[1]
        
        if "input_tokens" in df.columns and "output_tokens" in df.columns:
            if "task" in df.columns:
                agg = df.groupby(["task", "prompt_type"])[["input_tokens", "output_tokens"]].mean().reset_index()
                agg["model"] = model_name
                token_data.append(agg)
            else:
                agg = df.groupby("prompt_type")[["input_tokens", "output_tokens"]].mean().reset_index()
                agg["model"] = model_name
                agg["task"] = "unknown"
                token_data.append(agg)
    except Exception as e:
        print(f"Could not process {file}: {e}")

if token_data:
    token_df = pd.concat(token_data, ignore_index=True)
    token_df["total_tokens"] = token_df["input_tokens"] + token_df["output_tokens"]
    
    # Plot average output tokens by Task and Style
    plt.figure(figsize=(14, 7))
    ax1 = sns.barplot(data=token_df, x="task", y="output_tokens", hue="prompt_type", errorbar=None)
    plt.title("Average Output Tokens per Generation by Task and Prompt Style (All Models)")
    plt.ylabel("Avg Output Tokens")
    plt.xlabel("Task")
    add_bar_labels(ax1, fmt='%d')
    plt.tight_layout()
    plt.show()

    # Plot average output tokens by Model and Style
    plt.figure(figsize=(14, 7))
    ax2 = sns.barplot(data=token_df, x="model", y="output_tokens", hue="prompt_type", errorbar=None)
    plt.title("Average Output Tokens per Generation by Model and Prompt Style")
    plt.ylabel("Avg Output Tokens")
    plt.xlabel("Model")
    plt.xticks(rotation=45)
    add_bar_labels(ax2, fmt='%d')
    plt.tight_layout()
    plt.show()
else:
    print("No token data found in raw files.")


### Conclusion & Next Steps
1. **CrystalBLEU vs Jaccard**: Compare these two metrics to see if models are getting the *vocabulary* right (Jaccard) even if the *structure* (CrystalBLEU) is different.
2. **Cost-Benefit**: Is CoT significantly better than Few-Shot to justify the massive increase in output tokens?
3. **GT Baseline**: Emphasize cases where `Avg Violations` < `GT Avg Violations` in your paper to defend against Reviewer 148A.